# Set GPU Device

In [ ]:
import torch
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# 測試是否有使用GPU
if not torch.cuda.is_available():
    print('無法使用CUDA')
else:
    print(f'正在使用{torch.cuda.get_device_name(0)}')

# Check whether related file exists

In [ ]:
import os

folder_path = './plot'

if os.path.exists(folder_path):
    print(f"列出資料夾內容: {folder_path}")
    for filename in os.listdir(folder_path):
        print(" -", filename)
else:
    print(f"資料夾不存在: {folder_path}")


In [ ]:
import joblib
try:
    pca = joblib.load('./plot/ipca_model.pkl')
    print("成功載入 PCA 模型")
except Exception as e:
    print(f"PCA 載入失敗：{e}")

# Import Library

In [ ]:
import os
import sys
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import torch

sys.path.append('./base')
sys.path.append('./classifer')
sys.path.append('./edm2impl')

from base.generate_images import config_presets
from base.torch_utils import distributed as dist
from edm2impl.generate_images_for_test import generate_images_complete
from edm2impl.utils import load_network, load_classifier


In [ ]:
dist.init()

In [ ]:
preset_id = 'edm2-img512-s-guid-dino'
net, gnet, encoder = load_network(config_presets[preset_id].net, config_presets[preset_id].gnet,device=device)

In [ ]:
title = ''
print(torch.__version__)

# Hyperparameters for network

In [ ]:
title += preset_id
outdir = './out'                             # 圖片輸出的位置
subdirs = False
# seeds = [42]
# seeds = [42, 43, 44, 45]
seeds = [42, 43, 44, 45,46,47,48,49,50,51,52,53,54,55,56]
# class_idx = [0,0,0,0]  
class_idx = None
max_batch_size = 4                         # 一次餵給network生成的圖片數量
encoder_batch_size = 2                       # 一次餵給encoder做encode的圖片數量
verbose = True

# Hyperparameters for sampling

In [ ]:
num_steps = 32
sigma_min = 0.002
sigma_max = 80
rho = 7
guidance = None                                  # Guidance Scale, 可搭配scheduler 若填None則使用預設scheduler
S_churn = 0
S_min = 0
S_max = float('inf')
S_noise = 1

# Select the sampler 

In [ ]:
sampler_fn = 'map_neg_ddg'
title += ('+'+sampler_fn)
if sampler_fn == 'cfg':
    pass
elif sampler_fn == 'random_ddg':
    pass
elif sampler_fn == 'time_offset':
    pass
elif sampler_fn == 'cfg':
    pass
elif sampler_fn == 'noguid':
    pass
elif sampler_fn == 'map_neg_ddg':
    pass
else:
    raise ValueError(f"Unsupported sampler_fn: {sampler_fn}")
    

# Select the scheduler

In [ ]:
guidance_scheduler = 'trapezoid_scheduler'
title += ('+'+guidance_scheduler)
if guidance_scheduler == 'const_scheduler':
    pass
elif guidance_scheduler == 'linear_increase_scheduler':
    pass
elif guidance_scheduler == 'linear_decrease_scheduler':
    pass
elif guidance_scheduler == 'late_activate_scheduler':
    pass
elif guidance_scheduler == 'mid_activate_scheduler':
    pass
elif guidance_scheduler == 'interval_scheduler':
    pass
elif guidance_scheduler== 'trapezoid_scheduler':
    pass
else:
    raise ValueError(f"Unsupported guidance_scheduler: {guidance_scheduler}")
    

In [ ]:
DEBUG = False                             # 是否印出NaN意外與guidance scale變化

In [ ]:
pca_plot = True

# Parse arguements

In [ ]:
PARAMS = {
    'net': net,
    'gnet': gnet,
    'sampler':sampler_fn,
    'encoder': encoder,
    'seeds': seeds,
    'outdir': outdir,
    'class_idx': class_idx,
    'max_batch_size': max_batch_size,
    'num_steps':num_steps,
    'guidance_scheduler': guidance_scheduler,
    'debug': DEBUG,
    'pca_plot':pca_plot,
    'S_churn':S_churn,
    'device':device,
}

# Generate images

In [ ]:
# test_delta = [0,0.0001,0.0003,0.0007,0.001,0.002,0.003,0.005,0.007,0.01,0.02,0.03,0.04,0.05,0.07,0.1,0.15,0.2,0.3,0.4,0.5]
# test_delta = [2,3,4,5,6,7,8]
test_delta=[0]
# test_delta = [1,1.2,1.4,1.6,1.8,2.0,2.2,2.4,2.6]
# test_delta = [-0.05,-0.04,-0.03,-0.02,-0.01,0,0.01,0.02,0.03,0.04,0.05,0.06,0.07]
# test_delta = [32,64,128,256,512]
outdir_origin = outdir
for delta in test_delta:
    PARAMS['outdir']= outdir_origin+f'/int/delta={delta}/'
    # PARAMS['num_steps'] = delta
    # PARAMS['delta'] = delta
    PARAMS['low'] = 10
    PARAMS['low_peak'] = 20
    PARAMS['high_peak'] =25 
    PARAMS['high'] = 30
    PARAMS['n_random'] = 1
    # PARAMS['full_random'] = False
    # PARAMS['sigma_low'] = 0.28
    # PARAMS['class_idx'] = [320,320,320,320]
    # PARAMS['class_idx'] = [321,321,321,321]
    # PARAMS['negative_class_csv_path']='./plot/negative_class_manual.csv'
    PARAMS['negative_class_csv_path']='./plot/CN DDG CLIP v1.csv'
    PARAMS['k_average']=1
    PARAMS['guidance'] = 2.1
    
    PARAMS['debug'] = True
    PARAMS['heun_guid']=False
    # PARAMS['use_gnet']=False
    # PARAMS['delta_scheduler']='const_scheduler'
    # PARAMS['delta_scheduler']='linear_decrease_scheduler'
    # PARAMS['delta_scheduler']='S_scheduler'
    # PARAMS['delta_scheduler']='linear_increase_scheduler'
    PARAMS['classifier_use_scale_shift_norm'] = True
    PARAMS['classifier_resblock_updown'] = True
    # PARAMS['classifier_pool attention'] = True
    PARAMS['classifier_width'] = 128
    PARAMS['classifier_depth'] = 4
    PARAMS['classifier_attention_resolutions'] = '32,16,8'
    PARAMS['image_size'] = 64
    PARAMS['classifier_path'] = '/data/guidance-team-new/classifier_log_trained_by_train_noised_True/model100000.pt'
    image_iter = generate_images_complete(**PARAMS)
    images = []
    
    for result in image_iter:
        for img_tensor in result.images:
            img = img_tensor.permute(1, 2, 0).cpu().numpy()
            images.append(Image.fromarray(img))
    fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)
    
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        if i < len(images):
            ax.imshow(images[i])
            ax.axis('off')
        else:
            ax.axis('off')
    final_title = title + ',delta='+ str(delta)
    fig.suptitle(final_title, fontsize=16)
    plt.show()
    fig.savefig(outdir+"/"+final_title+'.png')
    # pca_image = Image.open(outdir+"/all_pca_trajectories.png")
    # draw = ImageDraw.Draw(pca_image)
    # draw.text((2500, 20), f"delta={delta}", fill="black",font=ImageFont.truetype("./plot/Roboto_Condensed-Black.ttf", size=60))
    # pca_image.save(f"{outdir}/pca_{title},delta={delta}.png")

# # 畫2*2結果圖gif
# image_paths = [f"{outdir}/pca_{title},delta={delta}.png" for delta in test_delta]
# images = [Image.open(p).convert('RGB') for p in image_paths]
# pca_gif_path = outdir+"/delta_pca_animation.gif"
# images[0].save(pca_gif_path,
#                save_all=True,
#                append_images=images[1:],
#                duration=500,     # 每幀持續時間 (ms)
#                loop=0)           # 0 = 無限循環
# print(f"GIF(PCA) saved to {pca_gif_path}")
# # 畫PCA Gif
# image_paths = [f"{outdir}/{title},delta={delta}.png" for delta in test_delta]
# images = [Image.open(p).convert('RGB') for p in image_paths]
# gif_path = outdir+"/delta_animation.gif"
# images[0].save(gif_path,
#                save_all=True,
#                append_images=images[1:],
#                duration=500,     # 每幀持續時間 (ms)
#                loop=0)           # 0 = 無限循環
# print(f"GIF saved to {gif_path}")

# Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)

axes = axes.flatten()
for i, ax in enumerate(axes):
    if i < len(images):
        ax.imshow(images[i])
        ax.axis('off')
    else:
        ax.axis('off')

fig.suptitle(title, fontsize=16)
plt.show()
# fig.savefig(f"{outdir}/{title}.png")

In [ ]:
img_path = outdir+'/all_pca_trajectories.png'

# 讀取圖片
img = Image.open(img_path)

# 顯示圖片
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.title('PCA Trajectories')
plt.show()

In [ ]:
# ! python -u generate_images_complete.py --preset=edm2-img512-xs-guid-dino \
# --outdir=out --subdirs --seeds=0-100 --guidance_scheduler=const_scheduler --sample_fn=edm_sampler_complete --batch=1

In [ ]:
# ! python calculate_metrics.py calc --images=out --ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl --metrics=fid


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# guidance 預設值
guidance = 1.7
num_steps = 32  # EDM2 中的 N
steps = np.arange(num_steps)

# 你提供的 scheduler 函數
def linear_increase_scheduler(i, num_steps, guidance, **kwargs):
    guidance = guidance - 1
    return 2 * (i / num_steps) * guidance + 1

def linear_decrease_scheduler(i, num_steps, guidance, **kwargs):
    guidance = guidance - 1
    return 2 * (1 - i / num_steps) * guidance + 1

def late_activate_scheduler(i, num_steps, guidance, **kwargs):
    guidance = guidance - 1
    return 2 * guidance + 1 if i > num_steps / 2 else 1

def mid_activate_scheduler(i, num_steps, guidance, **kwargs):
    guidance = guidance - 1
    return 2 * guidance + 1 if num_steps / 4 < i <= num_steps * 3 / 4 else 1

# scheduler 映射
config_guidance_scheduler = {
    'linear_increase_scheduler': linear_increase_scheduler,
    'linear_decrease_scheduler': linear_decrease_scheduler,
    'late_activate_scheduler': late_activate_scheduler,
    'mid_activate_scheduler': mid_activate_scheduler,
    'const_scheduler': None,
}

# helper: 將名稱變成標題格式
def format_title(name):
    return name.replace('_', ' ').title()

# 逐個繪製
for name, scheduler in config_guidance_scheduler.items():
    if scheduler is None:
        y = np.full(num_steps, guidance)
    else:
        y = np.array([scheduler(i, num_steps, guidance) for i in steps])
    
    plt.figure(figsize=(6, 4))
    plt.plot(steps, y, label=name)
    plt.title(f'{format_title(name)} (guidance={guidance})')
    plt.xlabel('Denoise Time')
    plt.ylabel('Guidance Scale')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
